# EX9 — Metoda Jakobiego: Warunki zbieżności

## Warunki teoretyczne

Podstawowym warunkiem zbieżności metody Jakobiego jest warunek na **promień spektralny** macierzy iteracji:

$$\rho(M) = \max_i |\lambda_i(M)| < 1$$

gdzie \(M\) to macierz iteracji metody Jakobiego, a \(\lambda_i(M)\) to jej wartości wąsne.

### Dominacja diagonalna

- Dla macierzy **mocno dominujących diagonalnie** metoda Jakobiego **zawsze zbiega**.
- Macierz \(A\) jest mocno dominująca diagonalnie, jeśli dla każdego wiersza \(i\):

$$|a_{ii}| > \sum_{j \neq i} |a_{ij}|$$

- Jeśli \(A\) jest macierzą **nieredukowalną** i **słabo dominującą diagonalnie** (z nierównością mocną przynajmniej dla jednego wiersza), metoda Jakobiego również zbiega.

### Macierz iteracji

Podział macierzy \(A\) na składowe:

$$A = D - L - U$$

- \(D\) — postać diagonalna z \(A\)
- \(L\) — część striccie dolna (z mínusem w konwencji podziału)
- \(U\) — część striccie górna

Macierz iteracji:

$$M = D^{-1}(L + U) = -D^{-1}(L + U)$$

Wektor wolny:

$$w = D^{-1}b$$

Krok iteracyjny:

$$x^{(k+1)} = M \, x^{(k)} + w$$

Im mniejszy promień spektralny \(\rho(M)\), tym szybsza zbieżność metody.


## $\boxed{\text{Eksperyment interaktywny}}$

W oryginalnym ścwiczeniu MATLAB zmiennymi sterującymi (`Interactive Controls`) można zmieniać elementy macierzy na żywo. W tym notatniku Jupyter wykonaj samodzielnie poniższe eksperymenty:

1. **Zmień elementy diagonalne** macierzy \(A\) (np. zmniejsz wartości na przekątnej) i zaobserwuj:
   - Jak zmienia się promień spektralny \(\rho(M)\)
   - Czy metoda przestaje zbiegać (powstan\(\rho(M) > 1\))

2. **Przybliżenia zbieżności**: Porównaj liczbę iteracji potrzebnych do osiągnięcia dokładności dla różnych wartości \(\rho(M)\). Zwróć uwagę, że przy \(\rho(M)\) bliskim \(1\) metoda zbiega bardzo powoli, nawet jeśli warunek zbieżności jest spełniony.

3. **Destabilizacja**: Spróbuj "zpsuć" dominację diagonalną, zmieniając elementy dodatkowe — np. zwiększ \(a_{1,2}\) lub zmniejsz \(a_{1,1}\). Uruchom ponownie komórkę z kodem i zaobserwuj wynik.

> $\unicode{1F4A1}$ **Wskazówka**: W Jupyterze możesz ponownie uruchomić komórkę z kodem ($\text{Shift} + \text{Enter}$), aby uzyskać nowy wykres i porównać krzywe błędu — odpowiednik MATLABowskiego `$\texttt{hold on}$`.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# --- Macierz silnie dominująca diagonalnie ---
A = np.array([[115, -2, 3, 0, 6],
              [-3, 19, 1, -2, 7.4],
              [2, -1, -157, 1, 6.7],
              [4, 3, -5, 117, 9],
              [2, 3.5, 6.1, -4, -118.1]])

b = np.array([-1, 2, 3, 0.5, 43.1])

# --- Rozkład macierzy ---
D = np.diag(np.diag(A))
L = np.tril(A) - D
U = np.triu(A) - D

# --- Macierz iteracji M i wektor w ---
M = -np.linalg.inv(D) @ (L + U)
w = np.linalg.inv(D) @ b

# --- Promień spektralny ---
spectrumM = np.max(np.abs(np.linalg.eigvals(M)))
print(f"Promień spektralny macierzy M: {spectrumM:.6f}")
print(f"Warunek zbieżności: {'Spełniony — ρ(M) < 1' if spectrumM < 1 else 'NIE spełniony — ρ(M) >= 1'}")


# --- Iteracja Jakobiego ---
x0 = np.random.rand(5)
delta0 = 1e-8
i_max = 50
x_old = x0.copy()
err = []
n_iter = []
i = 1
error = 5 * delta0

while (error >= delta0) and (i < i_max):
    x_new = M @ x_old + w
    error = np.linalg.norm(x_new - x_old)
    x_old = x_new
    err.append(error)
    n_iter.append(i)
    i += 1

print(f"Liczba wykonanych iteracji: {len(n_iter)}")
print(f"Ostateczny błąd: {err[-1]:.2e}")


# --- Wykres semilogarytmiczny ---
plt.figure(figsize=(8, 5))
plt.semilogy(n_iter, err, "bo-", linewidth=1.5, markersize=5, label="Błąd |x^(k+1) - x^(k)|")
# W MATLAB: hold on — tutaj re-run komórki, aby nałożyć nowy wykres
plt.axhline(y=delta0, color="r", linestyle="--", linewidth=1, label=f"Próg δ = $10^{{-8}}$")
plt.xlabel("Numer iteracji")
plt.ylabel("Błąd (skala logarytmiczna)")
plt.title("Zbieżność metody Jakobiego")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np

A = np.array([[115, -2, 3, 0, 6],
              [-3, 19, 1, -2, 7.4],
              [2, -1, -157, 1, 6.7],
              [4, 3, -5, 117, 9],
              [2, 3.5, 6.1, -4, -118.1]])

b = np.array([-1, 2, 3, 0.5, 43.1])

# --- Dokładne rozwiązanie ---
x_exact = np.linalg.solve(A, b)
print("Rozwiązanie dokładne:")
print(np.array2string(x_exact, precision=8, separator=", "))

# --- Residuum ---
residual = A @ x_exact - b
print(f"\nResiduum (|Ax - b|): {np.linalg.norm(residual):.2e}")

print(f"\nMaksymalna wartość bezwzgldna residuum: {np.max(np.abs(residual)):.2e}")

## Wykorzystywane funkcje

| Funkcja / wyrażenie | Opis |
|---|---|
| `np.diag(A)` | Wyodrębnia diagonalę macierzy \(A\) |
| `np.diag(vec)` | Tworzy macierz diagonalną z wektora |
| `np.tril(A)` | Część dolna macierzy \(A\) (wraz z diagonalą) |
| `np.triu(A)` | Część górna macierzy \(A\) (wraz z diagonalą) |
| `np.linalg.inv(D)` | Macierz odwrotna do \(D\) |
| `np.linalg.eigvals(M)` | Wartości wąsne macierzy \(M\) |
| `np.linalg.norm(v)` | Norma \(L_2\) wektora |
| `np.linalg.solve(A, b)` | Rozwiązanie układu \(Ax = b\) |
| `np.random.rand(n)` | Losowe liczby z \(U[0,1]\) |
| `plt.semilogy()` | Wykres z logarytmiczną ością \(y\) |
| `plt.axhline()` | Pozioma linia pomocnicza |

---

*Powiązane: EX8 — Metoda Jakobiego (iteracyjny solver).*
